# 06 · Exploratory data analysis
**Step 8:** bring the data from SQL into Python and run univariate, bivariate, multivariate,
trend, correlation and distribution analysis. Write what you observe under each chart.

In [ ]:
import sys, warnings
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

from src import config
pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")
sns.set_theme(style="whitegrid", palette="viridis")

# Where the raw CSVs live. To rehearse on generated test data instead, use:
#   RAW_DIR = config.SYNTHETIC_DIR   (run `python scripts/make_synthetic_data.py` first)
RAW_DIR = config.RAW_DIR

In [ ]:
from src.db import get_engine
from sqlalchemy import text

engine = get_engine()
print("Connected:", engine.url.render_as_string(hide_password=True))


def query(sql, params=None):
    with engine.connect() as conn:
        return pd.read_sql(text(sql), conn, params=params)

In [ ]:
of = query("SELECT * FROM order_features")
of["order_purchase_timestamp"] = pd.to_datetime(of["order_purchase_timestamp"])
valid = of[of["is_valid_sale"] == 1]
cf = query("SELECT * FROM customer_features")
sf = query("SELECT * FROM seller_features")
print(of.shape, cf.shape, sf.shape)

## Univariate analysis

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(13, 8))
sns.histplot(valid["total_order_value"], bins=50, ax=ax[0, 0]); ax[0, 0].set_title("Order value")
sns.histplot(np.log1p(valid["total_order_value"]), bins=50, ax=ax[0, 1]); ax[0, 1].set_title("log(1 + order value)")
sns.countplot(x="review_score", data=of.dropna(subset=["review_score"]), ax=ax[1, 0]); ax[1, 0].set_title("Review scores")
sns.histplot(of["delivery_days"].dropna(), bins=50, ax=ax[1, 1]); ax[1, 1].set_title("Delivery days")
plt.tight_layout(); plt.show()

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
valid["primary_category"].value_counts().head(12).plot.barh(ax=ax[0], color="#1b7f5c"); ax[0].set_title("Top categories (orders)"); ax[0].invert_yaxis()
of["order_status"].value_counts().plot.bar(ax=ax[1], color="#1b7f5c"); ax[1].set_title("Order status")
of["primary_payment_type"].value_counts().plot.bar(ax=ax[2], color="#1b7f5c"); ax[2].set_title("Payment method")
plt.tight_layout(); plt.show()

## Bivariate analysis

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(17, 5))
d = of.dropna(subset=["delivery_status", "review_score"])
sns.boxplot(data=d, x="delivery_status", y="review_score", ax=ax[0]); ax[0].set_title("Review score by delivery status")
top_cats = valid["primary_category"].value_counts().head(10).index
sns.boxplot(data=valid[valid["primary_category"].isin(top_cats)], y="primary_category",
            x=np.log1p(valid["total_order_value"]), ax=ax[1]); ax[1].set_title("log order value by category")
sns.scatterplot(data=d.sample(min(5000, len(d)), random_state=0), x="delivery_delay_days",
                y="review_score", alpha=0.2, ax=ax[2]); ax[2].set_title("Delay (days) vs review score")
plt.tight_layout(); plt.show()

## Multivariate analysis

In [ ]:
cols = ["total_order_value", "freight_value", "n_items", "delivery_days", "delivery_delay_days", "review_score"]
sample = valid[cols].dropna().sample(min(3000, len(valid[cols].dropna())), random_state=0)
sns.pairplot(sample, corner=True, plot_kws={"alpha": 0.3, "s": 12})
plt.show()

In [ ]:
pivot = valid.dropna(subset=["review_score"]).pivot_table(
    index="customer_state", columns="delivery_status", values="review_score", aggfunc="mean")
sns.heatmap(pivot, annot=True, fmt=".2f", cmap="viridis"); plt.title("Mean review score: state x delivery status")
plt.show()

## Trend analysis

In [ ]:
monthly = valid.groupby("order_month").agg(revenue=("total_order_value", "sum"),
                                           orders=("order_id", "count")).reset_index()
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(monthly["order_month"], monthly["revenue"], marker="o", color="#1b7f5c"); ax.set_ylabel("Revenue")
ax2 = ax.twinx(); ax2.bar(monthly["order_month"], monthly["orders"], alpha=0.25, color="#d97706"); ax2.set_ylabel("Orders")
plt.xticks(rotation=60); plt.title("Monthly revenue and orders"); plt.show()

## Correlation analysis

In [ ]:
num = of[["total_order_value", "freight_value", "n_items", "max_installments", "delivery_days",
          "delivery_delay_days", "review_score", "customer_order_count"]].astype(float)
corr = num.corr(method="spearman")
sns.heatmap(corr, annot=True, fmt=".2f", cmap="vlag", center=0); plt.title("Spearman correlation"); plt.show()
pairs = corr.where(np.triu(np.ones(corr.shape, dtype=bool), 1)).stack().sort_values(key=abs, ascending=False)
display(pairs.head(8).rename("spearman_rho"))

## Distribution analysis

In [ ]:
desc = valid[["total_order_value", "freight_value", "delivery_days"]].describe().T
desc["skew"] = valid[["total_order_value", "freight_value", "delivery_days"]].skew()
desc["kurtosis"] = valid[["total_order_value", "freight_value", "delivery_days"]].kurt()
display(desc)

## Data-driven summary of what the data shows

In [ ]:
import importlib.util
from src.reporting import load_queries_module, default_params, collect_insight_data
from src.insights import generate_insights

q = load_queries_module()
run_sql = lambda sql, params=None: query(sql, params)
data = collect_insight_data(run_sql, q, default_params(run_sql, q))
for i in generate_insights(data):
    print(f"[{i['area']}] {i['observation']}\n   -> {i['interpretation']}\n")